In [ ]:
!pip install ipywidgets qiskit qiskit-aer -q

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import random
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

game = {
    "round": 1,
    "score": 0,
    "eve_caught": 0,
    "total_rounds": 8,
}

game_output = widgets.Output()

style = """
<style>
.elliot-box { padding: 20px; border-radius: 15px; background: linear-gradient(135deg, #1a1a2e, #16213e); color: white; font-family: 'Courier New', monospace; }
.alice { color: #4fc3f7; font-size: 18px; }
.bob { color: #81c784; font-size: 18px; }
.elliot { color: #64b5f6; font-size: 18px; }
.eve { color: #ef5350; font-size: 18px; }
.safe { color: #66bb6a; font-size: 16px; }
.danger { color: #ff7043; font-size: 16px; }
.warning { color: #ffa726; font-size: 16px; }
.locked { color: #ffa726; font-size: 18px; }
.title { color: #64b5f6; font-size: 26px; font-weight: bold; }
.subtitle { color: #aaa; font-size: 14px; }
.instruction { background: #0d1117; padding: 12px; border-radius: 8px; margin: 8px 0; color: #ccc; font-size: 14px; border-left: 3px solid #64b5f6; }
.eve-dot { display: inline-block; width: 14px; height: 14px; background: #ef5350; border-radius: 50%; }
.qiskit-box { background: #0d1117; padding: 15px; border-radius: 8px; margin: 10px 0; color: #aaa; font-family: monospace; border-left: 3px solid #81c784; }
.mini-demo-box { background: #0d1117; padding: 12px; border-radius: 8px; margin: 10px 0; color: #ccc; font-family: monospace; border-left: 3px solid #64b5f6; font-size: 14px; }
</style>
"""
display(HTML(style))

def show_screen(message="", title=""):
    with game_output:
        clear_output(wait=True)
        html = f'<div class="elliot-box">'
        html += f'<div class="title">🔵 {title}</div>'
        html += f'<div class="subtitle">Elliot the Electron — BB84 Quantum Security Game</div>'
        html += f'<hr style="border-color: #333;">'
        html += message
        html += f'</div>'
        display(HTML(html))

def show_menu():
    msg = """
    <div style='text-align: center; padding: 20px;'>
        <div class='elliot' style='font-size: 22px;'>🔵 "Hi! I'm Elliot. I'm an electron."</div>
        <div style='font-size: 15px; color: #ccc; margin-top: 10px;'>
            "I live inside a quantum wire between Alice and Bob.<br>
            My best friend is <b>Flash</b>
            ⚡ — a photon who carries me at the speed of light.<br>
            Together, we deliver secret messages. Our enemy: <b>Eve</b> the eavesdropper."
        </div>
    </div>

    <div class='instruction'>
        <b>📖 THE STORY</b><br>
        Alice sends qubits. You (Elliot) choose a <b>Z-shield</b> or <b>X-shield</b> to protect them.<br>
        Eve hides in the wire, trying to steal secrets. If she picks the <b>WRONG</b> shield to measure — <span class='danger'>WE CATCH HER!</span>
    </div>

    <div class='instruction'>
        <b>🎯 THE RULES</b><br>
        • Alice prepares a random qubit: <b>|0⟩, |1⟩, |+⟩, or |−⟩</b><br>
        • You choose a <b>shield (Z or X)</b> to transmit it<br>
        • Eve randomly attacks and guesses a basis<br>
        • <span class='safe'>Eve guesses WRONG → TRACE LEFT → CAUGHT!</span><br>
        • <span class='warning'>Eve guesses RIGHT → Secret stolen (but she can't always win)</span><br>
        • <span class='safe'>No Eve → Safe delivery!</span>
        • <span style='color:#81c784;'>Every round shows live Qiskit output of the CNOT cloning attempt!</span>
    </div>

    <div class='instruction'>
        <b>🧠 WHAT YOU'LL LEARN</b><br>
        1️⃣ <b>Superposition:</b> A qubit can exist in a combination of basis states<br>
        2️⃣ <b>Measurement:</b> Observing a quantum state changes the system<br>
        3️⃣ <b>No-Cloning Theorem:</b> Unknown quantum states cannot be perfectly copied<br>
        4️⃣ <b>Diagonal Basis States:</b> The |+⟩ and |−⟩ states help reveal eavesdropping<br>
        5️⃣ <b>BB84 Protocol:</b> Quantum key distribution secured by quantum physics<br>
        6️⃣ <b>Entanglement:</b> Quantum systems can exhibit correlations stronger than classical systems
    </div>

    <div class='instruction'>
        <b>👤 MEET THE CHARACTERS</b><br>
        📡 <span class='alice'><b>Alice</b></span> — Sends the secret qubits<br>
        🔵 <span class='elliot'><b>Elliot</b></span> — You! The electron messenger<br>
        👤 <span class='eve'><b>Eve</b></span> — The eavesdropper hiding in the wire<br>
        📡 <span class='bob'><b>Bob</b></span> — Receives the qubits<br>
    </div>

    <div style='text-align: center; margin-top: 20px; color: #64b5f6; font-size: 16px;'>
        ⬆️ 🛡️ <b>Click Z-Shield or X-Shield above to begin!</b> 🛡️ ⬆️
    </div>
    """
    show_screen(msg, "THE SAFETY PASSAGE")

def run_mini_demo(alice_basis, alice_bit, alice_state):
    qc = QuantumCircuit(2, 2)
    if alice_basis == "Z":
        if alice_bit == 1:
            qc.x(0)
    else:
        qc.h(0)
        if alice_bit == 1:
            qc.z(0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    sim = AerSimulator()
    qc_transpiled = transpile(qc, sim)
    result = sim.run(qc_transpiled, shots=512).result()
    counts = result.get_counts()
    output = "<div class='mini-demo-box'>"
    output += f"<b>🔬 LIVE QISKIT: CNOT on {alice_state}</b><br><br>"
    for state in ['00', '01', '10', '11']:
        count = counts.get(state, 0)
        bar = "█" * (count // 10)
        output += f"   {state}: {count:3d} {bar}<br>"
    if alice_basis == "Z":
        output += f"<br>✅ Cloning APPEARS successful — but only for classical bits (orthogonal states).<br>"
        output += f"<span style='color:#ffa726;'>This does NOT violate No-Cloning. The theorem allows copying orthogonal states.</span><br>"
    else:
        zeros = counts.get('01', 0) + counts.get('10', 0)
        output += f"<br>❌ <b>Cloning FAILED!</b> Off-diagonal = {zeros}/512<br>"
        output += f"<span style='color:#ff7043;'>CNOT created ENTANGLEMENT (Bell state), not a copy. No-Cloning holds!</span><br>"
    output += "</div>"
    return output

def run_cloning_demo():
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    sim = AerSimulator()
    qc_transpiled = transpile(qc, sim)
    result = sim.run(qc_transpiled, shots=1024).result()
    counts = result.get_counts()
    output = "<div class='qiskit-box'>"
    output += "<b>🔬 NO-CLONING THEOREM: FULL DEMONSTRATION</b><br><br>"
    output += "<b>🧪 QISKIT SIMULATION — CNOT FAILS TO CLONE |+⟩</b><br>"
    output += "The most intuitive way to copy classical bits is using a CNOT gate. Let's try that on a quantum superposition (|+⟩):<br><br>"
    output += "<b>Circuit:</b> H on q0 → CNOT(q0,q1) → Measure<br><br>"
    output += "<b>📊 Results:</b><br>"
    for state in ['00', '01', '10', '11']:
        count = counts.get(state, 0)
        bar = "█" * (count // 20)
        output += f"   {state}: {count:4d} {bar}<br>"
    zeros = counts.get('01', 0) + counts.get('10', 0)
    total = sum(counts.values())
    output += f"<br>🔍 <b>Analysis:</b><br>"
    output += f"   |00⟩ + |11⟩ = ENTANGLEMENT (Bell state)<br>"
    output += f"   |01⟩ + |10⟩ = {zeros}/{total} (would be ~50% if cloning worked)<br>"
    output += f"<br>❌ <b>Cloning FAILED.</b> CNOT created an entangled Bell state between the qubits instead of an independent copy.<br><br>"
    output += "<b>📐 IMPORTANT SCIENTIFIC NOTE:</b><br>"
    output += "This circuit implementation is an <i>educational demonstration</i>, not a formal mathematical proof of the theorem. "
    output += "Showing that a CNOT gate fails only rules out this specific copying configuration on a |+⟩ state; it does not prove that no other combination of quantum gates could achieve cloning.<br><br>"
    output += "<b>📘 THE MATHEMATICAL PROOF (From Linearity & Unitarity):</b><br>"
    output += "To show that cloning is impossible universally, suppose a cloning operation U could copy any arbitrary unknown state:<br>"
    output += "   U(|ψ⟩|0⟩) = |ψ⟩|ψ⟩   and   U(|φ⟩|0⟩) = |φ⟩|φ⟩<br><br>"
    output += "Because quantum unitaries preserve inner products, taking the inner product of both equations forces:<br>"
    output += "   ⟨ψ|φ⟩ = ⟨ψ|φ⟩²<br><br>"
    output += "This constraint is mathematically valid <b>only</b> if ⟨ψ|φ⟩ = 0 or ⟨ψ|φ⟩ = 1. This means cloning only works if the states are completely identical or completely orthogonal. "
    output += "Therefore, no universal quantum operation can copy an unknown arbitrary quantum state.<br><br>"
    output += "📚 <b>Takeaway:</b> Eve is bound by the laws of linear algebra. Physics—not algorithm design—safeguards the BB84 protocol!"
    output += "</div>"
    return output

def play_round(basis_choice):
    if game["round"] > game["total_rounds"]:
        return
    msg = ""
    alice_basis = random.choice(["Z", "X"])
    alice_bit = random.choice([0, 1])
    if alice_basis == "Z":
        alice_state = "|0⟩" if alice_bit == 0 else "|1⟩"
    else:
        alice_state = "|+⟩" if alice_bit == 0 else "|−⟩"
    msg += f"<div style='background:#0d1117;padding:12px;border-radius:8px;margin:5px 0;'>"
    msg += f"📊 <b>Round {game['round']}/{game['total_rounds']}</b> | ✅ Safe: {max(0, game['score'])} | 🚨 Eve Caught: {game['eve_caught']}"
    msg += f"</div>"
    msg += f"<div class='alice'>📡 <b>Alice:</b> 'Elliot, here is <b>{alice_state}</b>' (Basis: <b>{alice_basis}</b>)</div>"
    msg += f"<div class='elliot'>🔵 <b>Elliot:</b> 'Raising <b>{basis_choice}</b>-shield! Hold on, Flash!'</div>"
    msg += f"<div style='text-align:center;color:#555;margin:5px 0;'>⚡💨 ⚡💨 ⚡💨 photon traveling...</div>"
    eve_attacks = random.random() < 0.5
    disturbed = False
    if eve_attacks:
        eve_basis = random.choice(["Z", "X"])
        msg += f"<div class='eve'>👤 <span class='eve-dot'></span> <b>Eve:</b> *jumps into wire* 'Trying <b>{eve_basis}</b>-basis!'</div>"
        if eve_basis == alice_basis:
            msg += f"<div class='warning'>😈 <b>Eve matched!</b> She reads the bit and re-sends. Undetected!</div>"
            msg += f"<div class='elliot'>🔵 'She got lucky this time...' 😐</div>"
        else:
            msg += f"<div class='danger'>🚨 <b>WRONG BASIS!</b> The qubit COLLAPSED! Eve left a measurable TRACE!</div>"
            msg += f"<div class='elliot'>🔵 'HA! You triggered the alarm, Eve!' 😎</div>"
            disturbed = True
            game["eve_caught"] += 1
    else:
        msg += f"<div class='safe'>✨ <b>Wire is clear!</b> No Eve this round.</div>"
        msg += f"<div class='elliot'>🔵 'Smooth sailing, Flash!' 😊</div>"
    bob_basis = alice_basis
    msg += f"<div class='bob'>📡 <b>Bob:</b> 'Receiving in <b>{bob_basis}</b>-basis. Thanks, Elliot!'</div>"
    if disturbed:
        msg += f"<div class='danger'>⚠️ <b>ERRORS DETECTED!</b> Alice & Bob compare notes — disturbance found! Message DISCARDED.</div>"
        game["score"] -= 1
    elif eve_attacks and not disturbed:
        msg += f"<div class='warning'>✅ Bob got the right bit — but Eve also has a copy. She won this round by luck.</div>"
    else:
        msg += f"<div class='safe'>✅ <b>PERFECT DELIVERY!</b> Secret is safe with Bob!</div>"
        game["score"] += 1
    msg += run_mini_demo(alice_basis, alice_bit, alice_state)
    game["round"] += 1
    if game["round"] > game["total_rounds"]:
        msg += f"<hr style='border-color:#333;'>"
        msg += f"<div class='title'>🏁 MISSION COMPLETE!</div>"
        msg += f"<div style='font-size:16px;text-align:center;'>"
        msg += f"✅ Safe deliveries: <b>{max(0, game['score'])}</b> / {game['total_rounds']}<br>"
        msg += f"🚨 Eve caught: <b>{game['eve_caught']}</b> times<br><br>"
        if game["eve_caught"] >= 2:
            msg += f"<div class='elliot'>🔵 'We caught Eve <b>{game['eve_caught']}</b> times! BB84 WINS!'</div>"
            msg += f"<div class='safe'>Physics protects our secrets. The No-Cloning Theorem is UNDEFEATED.</div>"
        elif game["eve_caught"] >= 1:
            msg += f"<div class='elliot'>🔵 'We caught her once. In real QKD, thousands of rounds guarantee detection.'</div>"
        else:
            msg += f"<div class='elliot'>🔵 'Eve was quiet today. But over many rounds, physics guarantees she slips up.'</div>"
        msg += f"<br><b>🎓 QUANTUM LESSONS LEARNED:</b><br>"
        msg += f"1. Superposition = |0⟩ AND |1⟩ simultaneously<br>"
        msg += f"2. Measurement destroys the quantum state<br>"
        msg += f"3. No-Cloning: CANNOT copy unknown quantum states<br>"
        msg += f"4. Diagonal states |+⟩, |−⟩ = Eve's TRAP<br>"
        msg += f"5. BB84 = security guaranteed by PHYSICS, not math<br>"
        msg += f"6. Entanglement = non-classical correlation system properties<br>"
        msg += f"</div>"
        msg += run_cloning_demo()
    show_screen(msg, f"Round {game['round']-1}")

btn_z = widgets.Button(description='🛡️ Z-Shield', button_style='info', layout=widgets.Layout(width='170px', height='50px'))
btn_x = widgets.Button(description='🛡️ X-Shield', button_style='warning', layout=widgets.Layout(width='170px', height='50px'))
btn_menu = widgets.Button(description='📖 Menu', button_style='primary', layout=widgets.Layout(width='100px', height='40px'))
btn_reset = widgets.Button(description='🔄 New Game', button_style='danger', layout=widgets.Layout(width='130px', height='40px'))
btn_qiskit = widgets.Button(description='🔬 Cloning Demo', button_style='success', layout=widgets.Layout(width='160px', height='40px'))

def on_z(b):
    try:
        if game["round"] <= game["total_rounds"]:
            play_round("Z")
    except Exception as e:
        show_screen(f"<div class='danger'>Error: {str(e)}</div>", "System Exception")

def on_x(b):
    try:
        if game["round"] <= game["total_rounds"]:
            play_round("X")
    except Exception as e:
        show_screen(f"<div class='danger'>Error: {str(e)}</div>", "System Exception")

def on_menu(b):
    try:
        game["round"] = 1
        game["score"] = 0
        game["eve_caught"] = 0
        show_menu()
    except Exception as e:
        show_screen(f"<div class='danger'>Error: {str(e)}</div>", "System Exception")

def on_reset(b):
    try:
        game["round"] = 1
        game["score"] = 0
        game["eve_caught"] = 0
        show_screen("""
        <div style='text-align:center;padding:30px;'>
            <div class='elliot' style='font-size:20px;'>🔵 "New game! Let's protect the wire!"</div>
            <div style='color:#ccc;margin-top:10px;'>🛡️ <b>Click Z-Shield or X-Shield above to begin!</b> 🛡️</div>
        </div>
        """, "Ready!")
    except Exception as e:
        show_screen(f"<div class='danger'>Error: {str(e)}</div>", "System Exception")

def on_qiskit(b):
    try:
        if game["round"] <= game["total_rounds"]:
            msg = """
            <div style='text-align: center; padding: 40px;'>
                <div class='locked'>🔒 <b>Demo Locked</b></div><br>
                <div style='color: #ccc; font-size: 16px;'>
                    Complete all <b>8 rounds</b> of your mission first!<br><br>
                    The full No-Cloning demonstration unlocks after you've<br>
                    helped Elliot deliver the secret messages.<br><br>
                    <span style='color:#81c784;'>But watch the live Qiskit output each round —<br>you're already seeing the physics in action!</span><br><br>
                    🛡️ <b>Click Z-Shield or X-Shield above to play!</b>
                </div>
            </div>
            """
            show_screen(msg, "Demo Locked")
        else:
            msg = run_cloning_demo()
            msg += "<br><div style='color:#ccc;'>Click <b>🔄 New Game</b> to play again.</div>"
            show_screen(msg, "No-Cloning Demonstration")
    except Exception as e:
        show_screen(f"<div class='danger'>Error: {str(e)}</div>", "Error")

btn_z.on_click(on_z)
btn_x.on_click(on_x)
btn_menu.on_click(on_menu)
btn_reset.on_click(on_reset)
btn_qiskit.on_click(on_qiskit)

display(widgets.HBox([btn_z, btn_x, btn_menu, btn_reset, btn_qiskit]))
display(game_output)

show_menu()